In [ ]:
import os

def print_tree(path, prefix=""):
    # Lista todos os itens do diretório
    items = sorted(os.listdir(path))
    for index, item in enumerate(items):
        full_path = os.path.join(path, item)
        connector = "├── " if index < len(items) - 1 else "└── "
        print(prefix + connector + item)

        # Se for diretório, faz recursão
        if os.path.isdir(full_path):
            new_prefix = prefix + ("│   " if index < len(items) - 1 else "    ")
            print_tree(full_path, new_prefix)

# Exemplo de uso:
print_tree("./mappo_mlp_sunt_bus")


├── MAPPO-energy  (0,0,0,1) - 1
│   ├── checkpoint_000200
│   │   ├── .is_checkpoint
│   │   ├── checkpoint-200
│   │   └── checkpoint-200.tune_metadata
│   ├── checkpoint_000253
│   │   ├── .is_checkpoint
│   │   ├── checkpoint-253
│   │   └── checkpoint-253.tune_metadata
│   ├── env_metrics.csv
│   ├── events.out.tfevents.1764012666.terumo
│   ├── params.json
│   ├── params.pkl
│   ├── progress.csv
│   └── result.json
├── MAPPO-energy  (0,0,0,1) - 2
│   ├── checkpoint_000200
│   │   ├── .is_checkpoint
│   │   ├── checkpoint-200
│   │   └── checkpoint-200.tune_metadata
│   ├── checkpoint_000253
│   │   ├── .is_checkpoint
│   │   ├── checkpoint-253
│   │   └── checkpoint-253.tune_metadata
│   ├── env_metrics.csv
│   ├── events.out.tfevents.1764013708.terumo
│   ├── params.json
│   ├── params.pkl
│   ├── progress.csv
│   └── result.json
├── MAPPO-energy  (0,0,0,1) - 3
│   ├── checkpoint_000200
│   │   ├── .is_checkpoint
│   │   ├── checkpoint-200
│   │   └── checkpoint-200.tune_metadata

In [1]:
import os
import pandas as pd

BASE_DIRS = {
    "MAPPO": "mappo_mlp_sunt_bus",
    "MAA2C": "maa2c_mlp_sunt_bus",
    "IA2C": "ia2c_mlp_sunt_bus"
}

OUTPUT_FILE = "reward_stats_grouped.csv"

REWARD_COL = "episode_reward_mean"


def find_experiments(base_dir):
    exps = []
    for root, dirs, files in os.walk(base_dir):
        if "progress.csv" in files:
            exps.append(root)
    return sorted(exps)


def extract_base_name(exp_name):
    """
    Ex.: "MAPPO-energy (0,0,0,1) - 3" → "MAPPO-energy (0,0,0,1)"
    """
    if " - " in exp_name:
        return exp_name.split(" - ")[0]
    return exp_name


# Coletar mean dos SEEDS
seed_stats = []

for algo, base_dir in BASE_DIRS.items():
    if not os.path.exists(base_dir):
        continue

    experiments = find_experiments(base_dir)

    for exp in experiments:
        exp_name = os.path.basename(exp)
        base_name = extract_base_name(exp_name)

        progress_file = os.path.join(exp, "progress.csv")

        try:
            df = pd.read_csv(progress_file)
        except:
            continue

        if REWARD_COL not in df.columns:
            continue

        reward_mean = df[REWARD_COL].mean()

        seed_stats.append({
            "algo": algo,
            "group": base_name,
            "seed": exp_name,
            "mean_reward": reward_mean
        })


# Agora AGRUPAR por group
stats_df = pd.DataFrame(seed_stats)

grouped = stats_df.groupby(["algo", "group"])["mean_reward"].agg(
    ["mean", "std", "count"]
).reset_index()

grouped["mean_std"] = grouped["mean"].round(3).astype(str) + " ± " + grouped["std"].round(3).astype(str)

grouped.to_csv(OUTPUT_FILE, index=False)

print("\nTabela agrupada salva em:", OUTPUT_FILE)
print(grouped)



Tabela agrupada salva em: reward_stats_grouped.csv
    algo                        group        mean  std  count        mean_std
0  MAPPO      MAPPO-energy  (0,0,0,1)   46.370540  0.0      5    46.371 ± 0.0
1  MAPPO         MAPPO-occ\t(1,0,0,0) -569.462603  0.0      5  -569.463 ± 0.0
2  MAPPO     MAPPO-occ-sync (1,0,1,0) -527.573488  0.0      5  -527.573 ± 0.0
3  MAPPO       MAPPO-occ-up (1,1,0,0)   17.184255  0.0      5    17.184 ± 0.0
4  MAPPO         MAPPO-sync (0,0,1,0)  -53.299973  0.0      5     -53.3 ± 0.0
5  MAPPO  MAPPO-sync-energy (0,0,1,1)   -3.863783  0.0      5    -3.864 ± 0.0
6  MAPPO          MAPPO-up\t(0,1,0,0)  674.437446  0.0      5   674.437 ± 0.0
7  MAPPO    MAPPO-up-energy (0,1,0,1)  355.525464  0.0      5   355.525 ± 0.0
8  MAPPO      MAPPO-up-sync (0,1,1,0)  303.241212  0.0      5   303.241 ± 0.0


In [ ]:
AUC

In [2]:
import os
import pandas as pd
import numpy as np

BASE_DIRS = {
    "MAPPO": "mappo_mlp_sunt_bus",
    "MAA2C": "maa2c_mlp_sunt_bus",
    "IA2C": "ia2c_mlp_sunt_bus"
}

OUTPUT_FILE = "auc_grouped.csv"

REWARD_COL = "episode_reward_mean"


def find_experiments(base_dir):
    exps = []
    for root, dirs, files in os.walk(base_dir):
        if "progress.csv" in files:
            exps.append(root)
    return sorted(exps)


def extract_base_name(exp_name):
    """
    Ex.: 'MAPPO-energy (0,0,0,1) - 3' → 'MAPPO-energy (0,0,0,1)'
    """
    if " - " in exp_name:
        return exp_name.split(" - ")[0]
    return exp_name


seed_auc_list = []

# -------------------------------------------------------
# PROCESSAR TODAS AS PASTAS E CALCULAR AUC POR SEED
# -------------------------------------------------------
for algo, base_dir in BASE_DIRS.items():

    if not os.path.exists(base_dir):
        print(f"[WARN] Base directory not found: {base_dir}")
        continue

    experiments = find_experiments(base_dir)

    for exp in experiments:
        exp_name = os.path.basename(exp)
        base_name = extract_base_name(exp_name)

        progress_file = os.path.join(exp, "progress.csv")

        try:
            df = pd.read_csv(progress_file)
        except:
            print(f"[WARN] Could not read {progress_file}")
            continue

        if REWARD_COL not in df.columns:
            print(f"[WARN] Column {REWARD_COL} not found in {exp}")
            continue

        rewards = df[REWARD_COL].dropna().values

        # --------- AUC via trapz ---------
        auc_value = np.trapz(rewards)

        seed_auc_list.append({
            "algo": algo,
            "group": base_name,
            "seed": exp_name,
            "auc": auc_value
        })


# -------------------------------------------------------
# AGRUPAR AUC POR SETUP
# -------------------------------------------------------
df = pd.DataFrame(seed_auc_list)

grouped = df.groupby(["algo", "group"])["auc"].agg(
    ["mean", "std", "count"]
).reset_index()

grouped["mean_std"] = grouped["mean"].round(3).astype(str) + " ± " + grouped["std"].round(3).astype(str)

# Salvar
grouped.to_csv(OUTPUT_FILE, index=False)

print(f"\n[OK] AUC agrupado gerado em: {OUTPUT_FILE}")
print(grouped)


[WARN] Base directory not found: maa2c_mlp_sunt_bus
[WARN] Base directory not found: ia2c_mlp_sunt_bus

[OK] AUC agrupado gerado em: auc_grouped.csv
    algo                        group           mean  std  count  \
0  MAPPO      MAPPO-energy  (0,0,0,1)   11915.541044  0.0      5   
1  MAPPO         MAPPO-occ\t(1,0,0,0) -283515.578740  0.0      5   
2  MAPPO     MAPPO-occ-sync (1,0,1,0) -131797.277838  0.0      5   
3  MAPPO       MAPPO-occ-up (1,1,0,0)    4586.762709  0.0      5   
4  MAPPO         MAPPO-sync (0,0,1,0)  -13041.226961  0.0      5   
5  MAPPO  MAPPO-sync-energy (0,0,1,1)    -663.067696  0.0      5   
6  MAPPO          MAPPO-up\t(0,1,0,0)  169152.030229  0.0      5   
7  MAPPO    MAPPO-up-energy (0,1,0,1)   89310.337442  0.0      5   
8  MAPPO      MAPPO-up-sync (0,1,1,0)   76217.588300  0.0      5   

            mean_std  
0    11915.541 ± 0.0  
1  -283515.579 ± 0.0  
2  -131797.278 ± 0.0  
3     4586.763 ± 0.0  
4   -13041.227 ± 0.0  
5     -663.068 ± 0.0  
6    1691

In [ ]:
CHARTS

In [8]:
# -*- coding: utf-8 -*-
"""
Gera gráficos de recompensa (normalizada e raw) AGRUPADOS por variação, ex.:

MAPPO-energy (0,0,0,1)
MAPPO-occ (1,0,0,0)
MAPPO-up-sync (0,1,1,0)
...

Cada variação usa os seeds:
MAPPO-energy (0,0,0,1) - 1
MAPPO-energy (0,0,0,1) - 2
...
"""

import os
import re
import glob
import argparse
from collections import defaultdict, Counter

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.python.summary.summary_iterator import summary_iterator
from scipy.ndimage import gaussian_filter1d

# =============================
# PASTAS DOS SEUS EXPERIMENTOS
# =============================
BASE_DIRS = [
    "mappo_mlp_sunt_bus",
    "maa2c_mlp_sunt_bus",
    "ia2c_mlp_sunt_bus"
]

# =============================
# FUNÇÕES AUXILIARES
# =============================
def get_event_files_in_dir(d):
    return sorted(glob.glob(os.path.join(d, "**", "events.out.tfevents.*"),
                            recursive=True))

def safe_iter_events(path):
    try:
        for e in summary_iterator(path):
            yield e
    except:
        pass

RAY_PREFIX = re.compile(r"^(ray/(tune|train|rllib)/)")
def strip_ray_prefix(tag): return RAY_PREFIX.sub("", tag)

REWARD_TAG_PREFERENCE = [
    "evaluation/episode_reward_mean",
    "episode_reward_mean",
    "rollout/episode_reward_mean",
]

REWARD_REGEX = re.compile(
    r"(episode|ep).*(reward|return).*(mean|avg)", re.IGNORECASE
)

X_AXIS_TAG_PREFERENCE = [
    "timesteps_total",
    "training_iteration",
]

def choose_reward_tag(available):
    for t in REWARD_TAG_PREFERENCE:
        if t in available:
            return t
    for t in available:
        if REWARD_REGEX.search(t):
            return t
    return None

def choose_x_axis_tag(available):
    for t in X_AXIS_TAG_PREFERENCE:
        if t in available:
            return t
    return None


def extract_series(event_paths, y_tag, x_tag=None):
    def match(tag, wanted): return strip_ray_prefix(tag) == wanted
    x_agg = defaultdict(list) if x_tag else None
    y_agg = defaultdict(list)

    for p in event_paths:
        x_run, y_run = {}, {}
        for e in safe_iter_events(p):
            for v in e.summary.value:
                t = strip_ray_prefix(v.tag)
                if x_tag and t == x_tag:  x_run[e.step] = float(v.simple_value)
                if t == y_tag:            y_run[e.step] = float(v.simple_value)
        if x_tag:
            for s, xv in x_run.items(): x_agg[s].append(xv)
        for s, yv in y_run.items(): y_agg[s].append(yv)

    if not y_agg:
        return [], [], []

    steps = sorted(y_agg.keys())
    xs = [float(s) for s in steps] if not x_tag else \
         [np.mean(x_agg[s]) if s in x_agg else float(s) for s in steps]

    means = [np.mean(y_agg[s]) for s in steps]
    stds  = [np.std(y_agg[s]) for s in steps]

    order = np.argsort(xs)
    xs, means, stds = np.array(xs)[order], np.array(means)[order], np.array(stds)[order]
    return xs, means, stds


def fixed_linear_norm(v, vmin, vmax):
    return np.clip((v - vmin) / (vmax - vmin), 0, 1)


def plot_norm(xs, m, s, title, save):
    sm = gaussian_filter1d(m, sigma=3)
    ss = gaussian_filter1d(s, sigma=3)
    nm = fixed_linear_norm(sm, -650, 100)
    ns = ss / (100 - (-650))
    lower, upper = np.clip(nm-ns,0,1), np.clip(nm+ns,0,1)

    plt.figure(figsize=(12,6))
    plt.plot(xs, nm)
    plt.fill_between(xs, lower, upper, alpha=0.2)
    plt.ylim(0,1)
    plt.title(title)
    plt.grid()
    os.makedirs(os.path.dirname(save), exist_ok=True)
    plt.savefig(save, dpi=150)
    plt.close()


def plot_raw(xs, m, s, title, save):
    sm = gaussian_filter1d(m, sigma=3)
    ss = gaussian_filter1d(s, sigma=3)
    plt.figure(figsize=(12,6))
    plt.plot(xs, sm)
    plt.fill_between(xs, sm-ss, sm+ss, alpha=0.2)
    plt.title(title)
    plt.grid()
    os.makedirs(os.path.dirname(save), exist_ok=True)
    plt.savefig(save, dpi=150)
    plt.close()


# =============================
# AGRUPAMENTO POR VARIAÇÃO
# =============================
def extract_variation_name(dirname):
    # "MAPPO-energy (0,0,0,1) - 3" --> "MAPPO-energy (0,0,0,1)"
    return dirname.split(" - ")[0]


# =============================
# MAIN
# =============================
def main():
    for root in BASE_DIRS:
        if not os.path.isdir(root):
            print("Ignorando:", root)
            continue

        # Descobrir variações
        subdirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root,d))]
        variations = defaultdict(list)

        for d in subdirs:
            var = extract_variation_name(d)
            variations[var].append(os.path.join(root, d))

        # Processar cada variação
        for var, folders in variations.items():
            print("Processando:", var)

            # Coletar todos os EVENT FILES dos seeds
            event_files = []
            for f in folders:
                event_files.extend(get_event_files_in_dir(f))

            if not event_files:
                print("Sem eventos em:", var)
                continue

            # Detectar tags
            raw, stripped = tag_frequency(event_files)
            available = set(stripped.keys())
            y_tag = choose_reward_tag(available)
            x_tag = choose_x_axis_tag(available)

            xs, means, stds = extract_series(event_files, y_tag, x_tag)
            if len(xs) == 0:
                print("Sem dados válidos:", var)
                continue

            charts_dir = os.path.join(root, "charts", var)

            # Normalizado
            plot_norm(
                xs, means, stds,
                title=f"{var} — normalized",
                save=os.path.join(charts_dir, "normalized.png")
            )

            # Raw
            plot_raw(
                xs, means, stds,
                title=f"{var} — raw reward",
                save=os.path.join(charts_dir, "raw.png")
            )

            print("✔ OK:", var)


if __name__ == "__main__":
    # IMPORTANTE PARA NOTEBOOKS
    import sys
    if "ipykernel" in sys.argv[0]:
        argv = []
    else:
        argv = None

    main()


Processando: MAPPO-sync (0,0,1,0)
✔ OK: MAPPO-sync (0,0,1,0)
Processando: MAPPO-occ	(1,0,0,0)


/mnt/ssd1/rafael/tmp/ipykernel_1802329/1497568129.py:133: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.savefig(save, dpi=150)
/mnt/ssd1/rafael/tmp/ipykernel_1802329/1497568129.py:146: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.savefig(save, dpi=150)


✔ OK: MAPPO-occ	(1,0,0,0)
Processando: MAPPO-sync-energy (0,0,1,1)
✔ OK: MAPPO-sync-energy (0,0,1,1)
Processando: MAPPO-up-sync (0,1,1,0)
✔ OK: MAPPO-up-sync (0,1,1,0)
Processando: MAPPO-energy  (0,0,0,1)
✔ OK: MAPPO-energy  (0,0,0,1)
Processando: charts
Sem eventos em: charts
Processando: MAPPO-occ-up (1,1,0,0)
✔ OK: MAPPO-occ-up (1,1,0,0)
Processando: MAPPO-occ-sync (1,0,1,0)
✔ OK: MAPPO-occ-sync (1,0,1,0)
Processando: MAPPO-up	(0,1,0,0)
✔ OK: MAPPO-up	(0,1,0,0)
Processando: MAPPO-up-energy (0,1,0,1)
✔ OK: MAPPO-up-energy (0,1,0,1)
Ignorando: maa2c_mlp_sunt_bus
Ignorando: ia2c_mlp_sunt_bus
